# Banking: Credit Card Fraud Detection (Production-style Pipeline)

Highly imbalanced **transaction fraud** detection for issuers using OpenML credit card data and weighted ANN.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Business Context
Fraud causes direct losses; models must balance **false alarms** vs **missed fraud**.


In [ ]:
# OpenML credit card fraud (sklearn fetch — downloads from internet)
from sklearn.datasets import fetch_openml

print("Downloading OpenML creditcard dataset (may take a minute)...")
data = fetch_openml(data_id=1597, as_frame=True, parser="auto")
df = data.frame
print(df.shape)
df.head()


In [ ]:
# EDA — extreme imbalance
if "Class" in df.columns:
    y_col = "Class"
else:
    y_col = df.columns[-1]
df[y_col] = df[y_col].astype(int)
print(df[y_col].value_counts())
sns.countplot(data=df, x=y_col)
plt.title("Fraud (1) vs Normal (0)")
plt.show()


In [ ]:
# Subsample for notebook runtime (optional — comment out for full data)
# Use stratified sample to keep fraud cases
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, train_size=50000, random_state=SEED)
idx, _ = next(sss.split(df, df[y_col]))
df = df.iloc[idx].reset_index(drop=True)

X = df.drop(columns=[y_col]).values.astype(np.float32)
y = df[y_col].values


In [ ]:
# Train / validation / test split (stratified for classification)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp
)  # ~70/15/15

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "Val:", X_val_s.shape, "Test:", X_test_s.shape)


In [ ]:
# Class weight for imbalance
neg, pos = np.bincount(y_train.astype(int))
total = neg + pos
weight_for_1 = (total / (2.0 * pos)) if pos > 0 else 1.0
class_weight = {0: 1.0, 1: weight_for_1}
print("class_weight:", class_weight)

model = models.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["AUC"])
model.summary()


In [ ]:
checkpoint = callbacks.ModelCheckpoint(
    "ann_fraud_best.keras", monitor="val_loss", save_best_only=True, verbose=1
)
early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
)
cb_list = [checkpoint, early_stop, reduce_lr]


In [ ]:
cb_list  # defined above
history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=50,
    batch_size=256,
    class_weight=class_weight,
    callbacks=cb_list,
    verbose=1,
)
y_prob = model.predict(X_test_s, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d")
plt.title("Fraud detection confusion matrix")
plt.show()


In [ ]:
# Inference demo on a few test rows
loaded = keras.models.load_model("ann_fraud_best.keras")
sample_idx = np.arange(min(5, len(X_test_s)))
samples = X_test_s[sample_idx]
probs = loaded.predict(samples, verbose=0).ravel()
for i, p in zip(sample_idx, probs):
    print(f"Row {i} -> P(positive class): {p:.4f}, predicted: {int(p >= 0.5)}, actual: {int(y_test[i])}")

import joblib
joblib.dump(scaler, "scaler.pkl")
print("Saved scaler.pkl and ann_fraud_best.keras")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and class balance / fraud rate on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
